In [1]:
!pip install tensorflow opencv-python numpy matplotlib

# Image (face) input → Model → Output = emotion (Happy, Sad, Angry, etc.)

In [8]:
import numpy as np
import pandas as pd
import cv2                #Images ke liye (OpenCV)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout

# Load CSV dataset
data = pd.read_csv(r"C:\Users\Lenovo\Desktop\gaurav\fer2013.csv")

# Labels
labels = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']

faces = []
targets = []

# Convert pixels → image
for i in range(len(data)):
    pixels = data['pixels'][i]
    emotion = data['emotion'][i]

    face = np.array(pixels.split(), dtype='float32')
    face = face.reshape(48,48)

    faces.append(face)
    targets.append(emotion)

faces = np.array(faces) / 255.0
faces = faces.reshape(-1,48,48,1)
targets = np.array(targets)

# Build model
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(48,48,1)))     #Image se features nikalta hai (jese eyes, nose, edges)
model.add(MaxPooling2D(2,2))                   #Image size chhoti karta hai, Important info rakhta hai

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128,(3,3),activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())        #Image → 1D vector ban gaya

model.add(Dense(128,activation='relu'))      #Brain ki tarah decision lena start karta hai
model.add(Dropout(0.5))            #Overfitting kam karta hai ,Matlab model cheating na kare 😄

model.add(Dense(7,activation='softmax'))   #7 emotions me se ek choose karega

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy']) #optimizer → kaise seekhe (Adam = smart),loss → kitni galti ho rahi

# Train
model.fit(faces, targets, epochs=10, batch_size=64)

# Save model
model.save("emotion_model.h5")

print("✅ Model Trained & Saved!")

C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 75s 121ms/step - accuracy: 0.3164 - loss: 1.7042
Epoch 2/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 76s 135ms/step - accuracy: 0.4254 - loss: 1.4855
Epoch 3/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 80s 131ms/step - accuracy: 0.4693 - loss: 1.3835
Epoch 4/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 80s 128ms/step - accuracy: 0.5009 - loss: 1.3147
Epoch 5/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 70s 125ms/step - accuracy: 0.5188 - loss: 1.2662
Epoch 6/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 67s 119ms/step - accuracy: 0.5376 - loss: 1.2244
Epoch 7/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 67s 119ms/step - accuracy: 0.5509 - loss: 1.1848
Epoch 8/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 68s 121ms/step - accuracy: 0.5636 - loss: 1.1503
Epoch 9/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 69s 123ms/step - accuracy: 0.5720 - loss: 1.1231
Epoch 10/10
561/561 ━━━━━━━━━━━━━━━━━━━━ 72s 128ms/step - accuracy: 0.5828 - loss: 1.0953


✅ Model Trained & Saved!


In [9]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("emotion_model.h5")

# Labels
labels = ['Angry','Disgust','Fear','Happy','Sad','Surprise','Neutral']

# Face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")  #Ye Haar Cascade hai. Camera frame me face dhundta hai

# Start camera
cap = cv2.VideoCapture(0) #0 = default webcam ON camera start karega

print("Press ESC to exit")

while True:
    ret, frame = cap.read()       #camera se ek image (frame) liya
    
    if not ret:
        print("Camera not working")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #Color hata diya (fast processing ke liye)

    faces = face_cascade.detectMultiScale(gray, 1.3, 5) #Frame me jitne face hai unka location mil gaya

    for (x,y,w,h) in faces:            #Agar 2 face hai → dono detect honge
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (48,48))      #Face ka part cut kiya. 48x48 me convert (model ke according)
        face = face / 255.0
        face = face.reshape(1,48,48,1) #Model ko proper format me diya

        prediction = model.predict(face, verbose=0)
        label = labels[np.argmax(prediction)]
#sabse bada value.Uska label = emotion (e.g., Happy)
        cv2.rectangle(frame,(x,y),(x+w,y+h),(255,0,0),2)
        cv2.putText(frame,label,(x,y-10), #Face ke around box.Upar emotion likh diya
                    cv2.FONT_HERSHEY_SIMPLEX,0.9,(255,0,0),2)

    cv2.imshow("Emotion Detector", frame) #Screen pe live video dikhega

    if cv2.waitKey(1) & 0xFF == 27: #ESC dabane se band
        break

cap.release()
cv2.destroyAllWindows() #Camera OFF + window close

Press ESC to exit


# Camera ON → Face detect → Model ko image de → Emotion show1